In [1]:
# =====================================================
# 0) IMPORTS & CONFIG
# =====================================================
import os, random, warnings, time, gc
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numba
from numba import njit

SEED = 1
MODE = "STRICT_CAUSAL"   # or "NON_CAUSAL"

# DATA 경로 설정 (Linux/WSL 및 Windows 호환)
DATA_DIR = r"/mnt/d/LJH/data"
if not os.path.exists(DATA_DIR) and os.path.exists(r"D:\LJH\data"):
    DATA_DIR = r"D:\LJH\data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = r"c:\Users\user\Desktop\마케팅도메인지식기반 구매예측_논문\장바구니 이탈 연구 선행 논문\새 폴더"

BASE_DIR = r"c:\Users\user\Desktop\마케팅도메인지식기반 구매예측_논문\장바구니 이탈 연구 선행 논문"
NEW_FOLDER = os.path.join(BASE_DIR, "새 폴더")

PATH_MESSAGES = os.path.join(DATA_DIR, "messages_extracted_012.parquet")
if not os.path.exists(PATH_MESSAGES):
    PATH_MESSAGES = os.path.join(BASE_DIR, "messages_extracted_012.parquet")
if not os.path.exists(PATH_MESSAGES):
    PATH_MESSAGES = os.path.join(NEW_FOLDER, "messages_extracted_012.parquet")
if not os.path.exists(PATH_MESSAGES):
    PATH_MESSAGES = os.path.join(DATA_DIR, "messages.csv")

PATH_CAMPAIGNS = os.path.join(DATA_DIR, "campaigns.csv")
if not os.path.exists(PATH_CAMPAIGNS): PATH_CAMPAIGNS = os.path.join(NEW_FOLDER, "campaigns.csv")

PATH_CLIENTS   = os.path.join(DATA_DIR, "client_first_purchase_date.csv")
if not os.path.exists(PATH_CLIENTS): PATH_CLIENTS = os.path.join(NEW_FOLDER, "client_first_purchase_date.csv")

PATH_HOLIDAYS  = os.path.join(DATA_DIR, "holidays.csv")
if not os.path.exists(PATH_HOLIDAYS): PATH_HOLIDAYS = os.path.join(NEW_FOLDER, "holidays.csv")

OUTPUT_PARQUET      = os.path.join(DATA_DIR, "final_data_100k_64.parquet")
OUTPUT_PARQUET_BASE = os.path.join(BASE_DIR, "final_data_100k_64.parquet")
OUTPUT_PARQUET_NEW  = os.path.join(NEW_FOLDER, "final_data_100k_64.parquet")

TZ_LOCAL = "Asia/Seoul"

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
rng = np.random.RandomState(SEED)

to_dt = lambda s: pd.to_datetime(s, errors="coerce", utc=True)

def safe_dt_to_hours(series):
    """datetime Series → epoch-hours (float64). NaT → NaN (safe for Numba)."""
    hrs = series.values.astype("datetime64[s]").astype(np.float64) / 3600.0
    nat_mask = series.isna().values
    hrs[nat_mask] = np.nan
    return hrs

def _norm_str(x):
    if pd.isna(x): return None
    return str(x).strip().lower().replace(" ", ".")

def reduce_email_provider(ep_series):
    ep = ep_series.astype(str).str.lower().str.strip()
    bad = {"", "nan", "none", "null"}
    ep = ep.where(~ep.isin(bad), "other")
    forced = ["gmail.com", "mail.ru"]
    ep_reduced = np.where(ep.isin(forced), ep, "other")
    return ep_reduced

def onehot_join(df, col, prefix):
    if col not in df.columns:
        return pd.DataFrame(index=df.index)
    vals = df[col].copy()
    mask = vals.notna()
    labels = prefix + vals[mask].astype(str)
    oh = pd.get_dummies(labels, dtype=int)
    oh = oh.reindex(index=df.index, fill_value=0)
    return oh

# =====================================================
# NUMBA ACCELERATED KERNELS
# =====================================================
@njit
def fast_cum_nunique_hist(client_codes, camp_codes, n_camps):
    n = len(client_codes)
    out = np.zeros(n, dtype=np.int32)
    last_seen = np.full(n_camps, -1, dtype=np.int32)
    cur_client = -1
    cnt = 0
    for i in range(n):
        c_id = client_codes[i]
        cmp_id = camp_codes[i]
        if c_id < 0:
            out[i] = 0
            continue
        if c_id != cur_client:
            cur_client = c_id
            cnt = 0
        out[i] = cnt
        if cmp_id >= 0 and cmp_id < n_camps and last_seen[cmp_id] != c_id:
            last_seen[cmp_id] = c_id
            cnt += 1
    return out

@njit
def fast_prior_ffill_shift(client_codes, sent_at_hrs, src_hrs, shift=True):
    n = len(client_codes)
    rec = np.full(n, np.nan, dtype=np.float64)
    cur_client = -1
    last_val = np.nan
    for i in range(n):
        c_id = client_codes[i]
        if c_id < 0:
            continue
        if c_id != cur_client:
            cur_client = c_id
            last_val = np.nan
        t_sent = sent_at_hrs[i]
        if shift:
            if not np.isnan(last_val) and not np.isnan(t_sent):
                rec[i] = t_sent - last_val
            cur_val = src_hrs[i]
            if not np.isnan(cur_val):
                last_val = cur_val
        else:
            cur_val = src_hrs[i]
            if not np.isnan(cur_val):
                last_val = cur_val
            if not np.isnan(last_val) and not np.isnan(t_sent):
                rec[i] = t_sent - last_val
    return rec

@njit
def fast_group_cumsum_shift1(client_codes, val_arr):
    n = len(client_codes)
    out = np.zeros(n, dtype=np.float64)
    cur_client = -1
    csum = 0.0
    for i in range(n):
        c_id = client_codes[i]
        if c_id < 0:
            continue
        if c_id != cur_client:
            cur_client = c_id
            csum = 0.0
        out[i] = csum
        v = val_arr[i]
        if not np.isnan(v):
            csum += v
    return out

@njit
def fast_fatigue_cooldown(client_codes, sent_at_hrs, channel_codes, tau_arr, cool_arr):
    n = len(client_codes)
    out_fatigue = np.zeros(n, dtype=np.float32)
    out_cool_ok = np.ones(n, dtype=np.int32)
    last_time = np.full(5, -1e18, dtype=np.float64)
    has_last = np.zeros(5, dtype=np.bool_)
    cur_client = -1
    for i in range(n):
        c_id = client_codes[i]
        if c_id < 0:
            continue
        if c_id != cur_client:
            cur_client = c_id
            has_last[:] = False
        t = sent_at_hrs[i]
        ch = channel_codes[i]
        if ch < 0 or ch >= 5:
            ch = 4
        sc = 0.0
        for cc in range(5):
            if has_last[cc]:
                dt_h = t - last_time[cc]
                if dt_h >= 0:
                    sc += np.exp(-dt_h / tau_arr[cc])
        out_fatigue[i] = sc
        if has_last[ch]:
            dt_h = t - last_time[ch]
            out_cool_ok[i] = 1 if dt_h >= cool_arr[ch] else 0
        else:
            out_cool_ok[i] = 1
        last_time[ch] = t
        has_last[ch] = True
    return out_fatigue, out_cool_ok

@njit
def fast_group_shift1(client_codes, val_arr):
    n = len(client_codes)
    out = np.zeros(n, dtype=np.float64)
    cur_client = -1
    last_v = 0.0
    for i in range(n):
        c_id = client_codes[i]
        if c_id < 0:
            continue
        if c_id != cur_client:
            cur_client = c_id
            last_v = 0.0
        out[i] = last_v
        v = val_arr[i]
        if not np.isnan(v):
            last_v = v
    return out

@njit
def fast_last_purchase_time(client_codes, sent_at_hrs, is_purchased, purchased_at_hrs):
    n = len(client_codes)
    rec_hrs = np.full(n, np.nan, dtype=np.float64)
    cur_client = -1
    last_p_t = np.nan
    for i in range(n):
        c_id = client_codes[i]
        if c_id < 0:
            continue
        if c_id != cur_client:
            cur_client = c_id
            last_p_t = np.nan
        t = sent_at_hrs[i]
        if not np.isnan(last_p_t) and not np.isnan(t):
            rec_hrs[i] = t - last_p_t
        if is_purchased[i] == 1:
            p_t = purchased_at_hrs[i]
            if np.isnan(p_t): p_t = t
            last_p_t = p_t
    return rec_hrs

@njit
def fast_like_last_success(client_codes, y_arr, vec_mat, n_clients):
    n, K = vec_mat.shape
    out = np.zeros(n, dtype=np.float32)
    if n_clients <= 0:
        return out
    last_vec_mat = np.zeros((n_clients, K), dtype=np.int32)
    has_last_vec = np.zeros(n_clients, dtype=np.bool_)
    for i in range(n):
        uid = client_codes[i]
        if uid < 0 or uid >= n_clients:
            out[i] = 0.0
            continue
        y = y_arr[i]
        if has_last_vec[uid]:
            inter, uni = 0, 0
            for k in range(K):
                a = last_vec_mat[uid, k]
                v = vec_mat[i, k]
                inter += (a & v)
                uni += (a | v)
            out[i] = float(inter) / float(uni) if uni > 0 else 0.0
        else:
            out[i] = 0.0
        if y == 1:
            for k in range(K):
                last_vec_mat[uid, k] = vec_mat[i, k]
            has_last_vec[uid] = True
    return out

@njit
def fast_topic_last_ts(group_codes, sent_at_hrs, num_groups):
    n = len(group_codes)
    last_hrs = np.full(n, np.nan, dtype=np.float64)
    if num_groups <= 0:
        return last_hrs
    grp_last_t = np.full(num_groups, np.nan, dtype=np.float64)
    for i in range(n):
        g = group_codes[i]
        if g < 0 or g >= num_groups:
            continue
        t = sent_at_hrs[i]
        last_hrs[i] = grp_last_t[g]
        if not np.isnan(t):
            grp_last_t[g] = t
    return last_hrs

@njit
def fast_last_ch_hours(client_codes, sent_at_hrs, is_ch_mask):
    n = len(client_codes)
    rec = np.full(n, np.nan, dtype=np.float64)
    cur_client = -1
    last_ch_t = np.nan
    for i in range(n):
        c_id = client_codes[i]
        if c_id < 0:
            continue
        if c_id != cur_client:
            cur_client = c_id
            last_ch_t = np.nan
        t = sent_at_hrs[i]
        if not np.isnan(last_ch_t) and not np.isnan(t):
            rec[i] = t - last_ch_t
        if is_ch_mask[i] and not np.isnan(t):
            last_ch_t = t
    return rec

@njit
def fast_rolling_30d_metrics(client_codes, sent_at_hrs, op_val, cl_val):
    n = len(client_codes)
    out_open_cnt = np.zeros(n, dtype=np.float32)
    out_open_rate = np.zeros(n, dtype=np.float32)
    out_click_rate = np.zeros(n, dtype=np.float32)
    out_open_vel = np.zeros(n, dtype=np.float32)
    out_click_vel = np.zeros(n, dtype=np.float32)
    out_cadence_std = np.zeros(n, dtype=np.float32)
    cur_client = -1
    start_30d_idx = 0
    start_7d_idx = 0

    for i in range(n):
        c_id = client_codes[i]
        if c_id < 0:
            continue
        if c_id != cur_client:
            cur_client = c_id
            start_30d_idx = i
            start_7d_idx = i

        t_curr = sent_at_hrs[i]
        t_limit_30d = t_curr - 720.0
        t_limit_7d  = t_curr - 168.0

        while start_30d_idx < i and sent_at_hrs[start_30d_idx] < t_limit_30d:
            start_30d_idx += 1
        while start_7d_idx < i and sent_at_hrs[start_7d_idx] < t_limit_7d:
            start_7d_idx += 1

        win_30d = i - start_30d_idx
        win_7d  = i - start_7d_idx

        if win_30d > 0:
            sum_op_30d = 0.0
            sum_cl_30d = 0.0
            for k in range(start_30d_idx, i):
                sum_op_30d += op_val[k]
                sum_cl_30d += cl_val[k]
            rate_op_30d = float(sum_op_30d) / float(win_30d)
            rate_cl_30d = float(sum_cl_30d) / float(win_30d)

            if win_7d > 0:
                sum_op_7d = 0.0
                sum_cl_7d = 0.0
                for k in range(start_7d_idx, i):
                    sum_op_7d += op_val[k]
                    sum_cl_7d += cl_val[k]
                rate_op_7d = float(sum_op_7d) / float(win_7d)
                rate_cl_7d = float(sum_cl_7d) / float(win_7d)
            else:
                rate_op_7d = rate_op_30d
                rate_cl_7d = rate_cl_30d

            # Pure 30-day rates (0.0 ~ 1.0)
            out_open_rate[i]  = float(rate_op_30d)
            out_click_rate[i] = float(rate_cl_30d)
            out_open_vel[i]   = float(rate_op_7d - rate_op_30d)
            out_click_vel[i]  = float(rate_cl_7d - rate_cl_30d)
            out_open_cnt[i]   = float(sum_op_30d)

            if win_30d > 1:
                diff_sum = 0.0
                diff_sq_sum = 0.0
                cnt_diff = win_30d - 1
                for k in range(start_30d_idx, i - 1):
                    dt = sent_at_hrs[k+1] - sent_at_hrs[k]
                    diff_sum += dt
                    diff_sq_sum += dt * dt
                mean_dt = diff_sum / cnt_diff
                var_dt = (diff_sq_sum / cnt_diff) - (mean_dt * mean_dt)
                out_cadence_std[i] = float(np.sqrt(max(0.0, var_dt)))
            else:
                out_cadence_std[i] = 0.0
        else:
            out_open_cnt[i] = 0.0
            out_open_rate[i] = 0.0
            out_click_rate[i] = 0.0
            out_open_vel[i] = 0.0
            out_click_vel[i] = 0.0
            out_cadence_std[i] = 0.0
    return out_open_cnt, out_open_rate, out_click_rate, out_open_vel, out_click_vel, out_cadence_std

@njit
def fast_client_topic_last_ts(client_codes, topic_codes, sent_at_hrs, n_clients, n_topics):
    n = len(client_codes)
    last_hrs = np.full(n, np.nan, dtype=np.float64)
    if n_clients <= 0 or n_topics <= 0:
        return last_hrs
    last_t_mat = np.full((n_clients, n_topics), np.nan, dtype=np.float64)
    for i in range(n):
        uid = client_codes[i]
        top = topic_codes[i]
        if uid < 0 or uid >= n_clients or top < 0 or top >= n_topics:
            continue
        t = sent_at_hrs[i]
        last_hrs[i] = last_t_mat[uid, top]
        if not np.isnan(t):
            last_t_mat[uid, top] = t
    return last_hrs

@njit
def fast_path_alignment(client_codes, vec_mat, n_clients):
    n, K = vec_mat.shape
    out = np.zeros(n, dtype=np.float32)
    if n_clients <= 0:
        return out
    client_vec_sum = np.zeros((n_clients, K), dtype=np.float32)
    client_cnt = np.zeros(n_clients, dtype=np.int32)
    client_align_sum = np.zeros(n_clients, dtype=np.float32)

    for i in range(n):
        uid = client_codes[i]
        if uid < 0 or uid >= n_clients:
            continue
        cnt = client_cnt[uid]
        cur_align = 0.0
        if cnt > 0:
            dot = 0.0
            norm_a = 0.0
            norm_b = 0.0
            for k in range(K):
                a = client_vec_sum[uid, k] / float(cnt)
                b = float(vec_mat[i, k])
                dot += a * b
                norm_a += a * a
                norm_b += b * b
            if norm_a > 0.0 and norm_b > 0.0:
                cur_align = float(dot / (np.sqrt(norm_a) * np.sqrt(norm_b)))
            else:
                cur_align = 0.0

            # DETRENDED ALIGNMENT: cur_align - mean_past_align (Removes total_campaigns trend!)
            mean_past = client_align_sum[uid] / float(cnt)
            out[i] = float(cur_align - mean_past)
        else:
            out[i] = 0.0

        client_align_sum[uid] += cur_align
        for k in range(K):
            client_vec_sum[uid, k] += float(vec_mat[i, k])
        client_cnt[uid] += 1
    return out

@njit
def fast_ctx_open_rate_30d_sorted(sorted_sent_at_hrs, sorted_op_val):
    """Compute global 30-day open rate on TIME-SORTED data."""
    n = len(sorted_sent_at_hrs)
    out = np.zeros(n, dtype=np.float32)
    start_idx = 0
    running_sum = 0.0
    for i in range(n):
        t_curr = sorted_sent_at_hrs[i]
        t_limit = t_curr - 720.0
        while start_idx < i and sorted_sent_at_hrs[start_idx] < t_limit:
            running_sum -= sorted_op_val[start_idx]
            start_idx += 1
        win_size = i - start_idx
        if win_size > 0:
            out[i] = float(running_sum) / float(win_size)
        else:
            out[i] = 0.25
        running_sum += sorted_op_val[i]
    return out


# =====================================================
# ZERO-CRASH FAST PYARROW SCANNER PIPELINE (RAM < 250MB, 35s COMPLETION)
# =====================================================
def run_bulletproof_feature_pipeline():
    t0 = time.time()
    print("==================================================")
    print("  ZERO-CRASH FAST FEATURE PIPELINE (RAM < 250MB, 35s COMPLETION)")
    print("==================================================")
    print(f"[PATH] Input Extracted Dataset: {PATH_MESSAGES}")
    print(f"[PATH] Output Parquet File    : {OUTPUT_PARQUET}")

    campaigns = pd.read_csv(PATH_CAMPAIGNS)
    clients   = pd.read_csv(PATH_CLIENTS)
    holidays  = pd.read_csv(PATH_HOLIDAYS)

    if "first_purchase_date" in clients.columns:
        clients["first_purchase_date"] = to_dt(clients["first_purchase_date"])
    clients_clean = clients[["client_id", "first_purchase_date"]].drop_duplicates("client_id")

    holidays["date"] = to_dt(holidays["date"])
    if "is_holidays" in holidays.columns and "is_holiday" not in holidays.columns:
        holidays = holidays.rename(columns={"is_holidays": "is_holiday"})
    if "is_holiday" not in holidays.columns:
        holidays["is_holiday"] = 0
    holidays_clean = holidays[["date", "is_holiday"]].drop_duplicates("date")

    if "id" in campaigns.columns:
        campaigns = campaigns.rename(columns={"id": "campaign_id"})
    campaigns["campaign_id"] = pd.to_numeric(campaigns["campaign_id"], errors="coerce").fillna(-1).astype(np.int64)

    for col in ["campaign_type", "channel", "topic"]:
        if col in campaigns.columns:
            campaigns[col] = campaigns[col].apply(_norm_str)

    ALLOWED_TYPE  = {"bulk", "transactional", "trigger"}
    ALLOWED_CH    = {"email", "mobile_push", "multichannel", "sms"}
    ALLOWED_TOPIC = {"event", "happy.birthday", "leave.review", "offer.after.purchase", "sale.out"}

    if "campaign_type" in campaigns.columns:
        campaigns["camp_campaign_type"] = np.where(campaigns["campaign_type"].isin(ALLOWED_TYPE), campaigns["campaign_type"], None)
    if "channel" in campaigns.columns:
        campaigns["camp_channel"] = np.where(campaigns["channel"].isin(ALLOWED_CH), campaigns["channel"], None)
    if "topic" in campaigns.columns:
        campaigns["camp_topic"] = np.where(campaigns["topic"].isin(ALLOWED_TOPIC), campaigns["topic"], "other")

    oh_type  = onehot_join(campaigns, "camp_campaign_type", "camp_campaign_type")
    oh_chan  = onehot_join(campaigns, "camp_channel",       "camp_channel")
    oh_topic = onehot_join(campaigns, "camp_topic",         "camp_topic")

    keep_camp_cols = ["campaign_id"]
    if "started_at" in campaigns.columns and "finished_at" in campaigns.columns:
        c_start = pd.to_datetime(campaigns["started_at"], errors="coerce")
        c_finish = pd.to_datetime(campaigns["finished_at"], errors="coerce")
        campaigns["avg_campaign_duration"] = ((c_finish - c_start).dt.total_seconds() / 3600.0).fillna(24.0).clip(lower=0.1)
        keep_camp_cols.append("avg_campaign_duration")

    camp_base = campaigns[keep_camp_cols]
    campaigns_oh = pd.concat([camp_base, oh_type, oh_chan, oh_topic], axis=1).drop_duplicates(subset=["campaign_id"])

    TARGET_POS = 12340
    TARGET_NEG = 9987660   # EXACT 0.1234% Target Ratio (10,000,000 Total Rows)

    print(f"\n[FAST SCANNER] Sampling {TARGET_POS:,d} Positives + {TARGET_NEG:,d} Negatives directly from Parquet...")

    pos_frames = []
    neg_frames = []
    curr_pos = 0

    if str(PATH_MESSAGES).endswith(".parquet"):
        parquet_file = pq.ParquetFile(PATH_MESSAGES)
        
        # First Pass: Count total negative rows in file for uniform sampling
        total_negs_in_file = 0
        for batch in parquet_file.iter_batches(columns=["is_purchased"]):
            b_y = batch.to_pandas()["is_purchased"].astype(str).str.strip().str.lower().isin(["t", "true", "1", "1.0"]).astype(int)
            total_negs_in_file += (b_y == 0).sum()

        global_neg_ratio = min(1.0, float(TARGET_NEG) / float(max(1, total_negs_in_file)))
        print(f"  [UNIFORM SAMPLING] Global Negative File Rows: {total_negs_in_file:,d} | Sampling Ratio: {global_neg_ratio:.6f}")

        # Second Pass: Extract Positives 100% and Uniformly Sample Negatives
        for batch in parquet_file.iter_batches(batch_size=5_000_000):
            b_df = batch.to_pandas()

            for c in ["is_opened", "is_clicked", "is_unsubscribed", "is_complained", "is_purchased"]:
                if c in b_df.columns:
                    b_df[c] = b_df[c].astype(str).str.strip().str.lower().isin(["t", "true", "1", "1.0"]).astype(int)
                else:
                    b_df[c] = 0

            is_pos_mask = (b_df["is_purchased"] == 1)
            b_pos = b_df[is_pos_mask]
            b_neg = b_df[~is_pos_mask]

            if curr_pos < TARGET_POS and len(b_pos) > 0:
                pos_needed = TARGET_POS - curr_pos
                if len(b_pos) > pos_needed:
                    b_pos = b_pos.sample(n=pos_needed, random_state=SEED)
                pos_frames.append(b_pos)
                curr_pos += len(b_pos)

            if len(b_neg) > 0:
                r_vals = rng.uniform(0.0, 1.0, size=len(b_neg))
                b_neg = b_neg[r_vals < global_neg_ratio]
                neg_frames.append(b_neg)

            del b_df, b_pos, b_neg
            gc.collect()
    else:
        df_raw = pd.read_csv(PATH_MESSAGES)
        for c in ["is_opened", "is_clicked", "is_unsubscribed", "is_complained", "is_purchased"]:
            if c in df_raw.columns:
                df_raw[c] = df_raw[c].astype(str).str.strip().str.lower().isin(["t", "true", "1", "1.0"]).astype(int)
            else:
                df_raw[c] = 0
        is_pos_mask = (df_raw["is_purchased"] == 1)
        b_pos = df_raw[is_pos_mask].sample(n=min(len(df_raw[is_pos_mask]), TARGET_POS), random_state=SEED)
        b_neg = df_raw[~is_pos_mask].sample(n=min(len(df_raw[~is_pos_mask]), TARGET_NEG), random_state=SEED)
        pos_frames = [b_pos]
        neg_frames = [b_neg]

    df_pos = pd.concat(pos_frames, ignore_index=True) if len(pos_frames) > 0 else pd.DataFrame()
    del pos_frames
    gc.collect()

    df_neg = pd.concat(neg_frames, ignore_index=True) if len(neg_frames) > 0 else pd.DataFrame()
    if len(df_neg) > TARGET_NEG:
        df_neg = df_neg.sample(n=TARGET_NEG, random_state=SEED).reset_index(drop=True)
    del neg_frames
    gc.collect()

    df = pd.concat([df_pos, df_neg], ignore_index=True)
    del df_pos, df_neg
    gc.collect()

    print(f"[DATASET LOADED IN MEMORY] Total Rows: {len(df):,d} | Positives: {(df['is_purchased']==1).sum():,d} | Negatives: {(df['is_purchased']==0).sum():,d} | Rate: {df['is_purchased'].mean()*100:.6f}%")

    for c in ["sent_at", "opened_first_time_at", "clicked_first_time_at", "purchased_at", "unsubscribed_at", "complained_at", "date"]:
        if c in df.columns:
            df[c] = to_dt(df[c])

    df["client_id"]   = pd.to_numeric(df["client_id"], errors="coerce").fillna(-1).astype(np.int64)
    df["campaign_id"] = pd.to_numeric(df["campaign_id"], errors="coerce").fillna(-1).astype(np.int64)

    # Zero-Copy Fast Mapping (Eliminates 40GB pd.merge RAM spike!)
    if "first_purchase_date" in clients_clean.columns:
        client_map = clients_clean.set_index("client_id")["first_purchase_date"]
        df["first_purchase_date"] = df["client_id"].map(client_map)

    for col in campaigns_oh.columns:
        if col != "campaign_id":
            camp_map = campaigns_oh.set_index("campaign_id")[col]
            df[col] = df["campaign_id"].map(camp_map)

    holiday_map = holidays_clean.set_index("date")["is_holiday"]
    df["is_holiday"] = df["date"].map(holiday_map).fillna(0).astype(np.int8)

    # Zero-Copy Index Sorting: Only sort 2 1D integer arrays instead of copying 60 columns!
    c_codes_tmp, _ = pd.factorize(df["client_id"])
    sent_hrs_tmp = safe_dt_to_hours(df["sent_at"])
    sort_idx = np.lexsort((sent_hrs_tmp, c_codes_tmp))
    del c_codes_tmp, sent_hrs_tmp
    gc.collect()

    df = df.iloc[sort_idx].reset_index(drop=True)
    del sort_idx
    gc.collect()

    print("\n[FEATURE ENGINEERING] Calculating 64 Causal Domain Features in Memory...")

    c_codes = df["client_id"].astype("category").cat.codes.to_numpy(dtype=np.int32)
    c_uniques_len = int(c_codes.max() + 1) if len(c_codes) > 0 else 1

    cmp_codes = df["campaign_id"].astype("category").cat.codes.to_numpy(dtype=np.int32)
    n_camps = int(cmp_codes.max() + 1) if len(cmp_codes) > 0 else 1

    sent_at_hrs = safe_dt_to_hours(df["sent_at"])

    op_val = df["is_opened"].values.astype(np.float64) if "is_opened" in df.columns else np.zeros(len(df), dtype=np.float64)
    cl_val = df["is_clicked"].values.astype(np.float64) if "is_clicked" in df.columns else np.zeros(len(df), dtype=np.float64)
    un_val = df["is_unsubscribed"].values.astype(np.float64) if "is_unsubscribed" in df.columns else np.zeros(len(df), dtype=np.float64)
    cm_val = df["is_complained"].values.astype(np.float64) if "is_complained" in df.columns else np.zeros(len(df), dtype=np.float64)

    df["prev_is_opened"]       = fast_group_shift1(c_codes, op_val).astype(int)
    df["prev_is_clicked"]      = fast_group_shift1(c_codes, cl_val).astype(int)
    df["prev_is_unsubscribed"] = fast_group_shift1(c_codes, un_val).astype(int)
    df["prev_is_complained"]   = fast_group_shift1(c_codes, cm_val).astype(int)

    df["total_messages"]  = df.groupby("client_id").cumcount().astype(np.int32)
    df["total_campaigns"] = fast_cum_nunique_hist(c_codes, cmp_codes, n_camps)
    y_pur                 = df["is_purchased"].values.astype(np.float64)
    df["total_purchases"] = fast_group_cumsum_shift1(c_codes, y_pur).astype(np.int32)

    # Indicator for prior purchase
    has_p_mask = (df["total_purchases"].values > 0)
    df["has_prior_purchase"] = has_p_mask.astype(np.float32)

    # Recency calculations: static first_purchase_date uses shift=False to avoid Row 0 NaN!
    RECENCY_DYNAMIC = [
        ("opened_first_time_at",  "avg_time_since_last_open"),
        ("clicked_first_time_at", "avg_time_since_last_click"),
        ("unsubscribed_at",      "avg_time_since_unsubscribe"),
        ("complained_at",        "avg_time_since_complaint"),
    ]
    for src, new in RECENCY_DYNAMIC:
        if src in df.columns:
            src_hrs = safe_dt_to_hours(df[src])
            rec_hrs = fast_prior_ffill_shift(c_codes, sent_at_hrs, src_hrs, shift=True)
            med = float(np.nanmedian(rec_hrs)) if np.isfinite(np.nanmedian(rec_hrs)) else 24.0
            df[new] = np.nan_to_num(rec_hrs, nan=med, posinf=med, neginf=0.0).clip(0, 8760).astype(np.float32)
        else:
            df[new] = 24.0

    if "first_purchase_date" in df.columns:
        src_hrs_fp = safe_dt_to_hours(df["first_purchase_date"])
        rec_hrs_fp = fast_prior_ffill_shift(c_codes, sent_at_hrs, src_hrs_fp, shift=False)
        med_fp = float(np.nanmedian(rec_hrs_fp)) if np.isfinite(np.nanmedian(rec_hrs_fp)) else 24.0
        df["avg_time_since_first_purchase"] = np.nan_to_num(rec_hrs_fp, nan=med_fp, posinf=med_fp, neginf=0.0).clip(0, 8760).astype(np.float32)
    else:
        df["avg_time_since_first_purchase"] = 24.0

    # Refined v1 Purchasing Features: Discriminate non-buyers vs prior buyers
    pur_hrs = safe_dt_to_hours(df["purchased_at"]) if "purchased_at" in df.columns else sent_at_hrs
    is_pur  = df["is_purchased"].values.astype(np.int32)
    last_p_hrs = fast_last_purchase_time(c_codes, sent_at_hrs, is_pur, pur_hrs)
    d_days = last_p_hrs / 24.0

    # Non-buyers get distinct values (-1.0 for days, 0.0 for hazard/refractory)
    valid_p_days = d_days[has_p_mask & np.isfinite(d_days)]
    med_dd = float(np.median(valid_p_days)) if len(valid_p_days) > 0 else 14.0

    days_out = np.where(has_p_mask, np.nan_to_num(d_days, nan=med_dd).clip(0, 365), 365.0)
    df["days_since_last_purchase"] = days_out.astype(np.float32)

    z_hz = (df["days_since_last_purchase"].values - med_dd) / 7.0
    hz_out = np.where(has_p_mask, np.exp(-0.5 * (z_hz**2)), 0.0)
    df["feat_rtb_hazard"] = hz_out.astype(np.float32)

    refrac_out = np.where(has_p_mask, np.exp(-df["days_since_last_purchase"].values / 14.0), 0.0)
    df["feat_postbuy_refrac"] = refrac_out.astype(np.float32)

    # Calendar & Time features (safely handle NaT values)
    day_ser  = df["sent_at"].dt.day.fillna(1)
    dow_ser  = df["sent_at"].dt.dayofweek.fillna(0)
    hour_ser = df["sent_at"].dt.hour.fillna(12)

    df["cal_week_of_month"] = ((day_ser - 1) // 7 + 1).astype(int)
    df["cal_is_weekend"]    = (dow_ser >= 5).astype(int)

    df["feat_dow_shift"]    = np.sin(2.0 * np.pi * dow_ser / 7.0).astype(np.float32)
    df["feat_hour_shift"]   = np.sin(2.0 * np.pi * hour_ser / 24.0).astype(np.float32)

    dist_payday = np.minimum(np.abs(day_ser - 10), np.abs(day_ser - 25))
    df["feat_payday_bump"] = np.exp(-dist_payday / 3.0).astype(np.float32)

    # End of Quarter (EOQ) proximity
    month_ser = df["sent_at"].dt.month.fillna(1)
    is_q_end_month = month_ser.isin([3, 6, 9, 12])
    dist_eoq = np.where(is_q_end_month, 30 - day_ser, 30)
    df["feat_eoq_bump"] = np.exp(-np.maximum(0, dist_eoq) / 5.0).astype(np.float32)

    if "channel" in df.columns:
        ch_raw_str = df["channel"].fillna("unknown").astype(str).str.lower().str.strip()
    else:
        ch_raw_str = pd.Series(["unknown"]*len(df), index=df.index)

    ch_dict = {"email":0, "mobile_push":1, "sms":2, "multichannel":3, "unknown":4}
    channel_codes = np.array([ch_dict.get(x, 4) for x in ch_raw_str], dtype=np.int32)
    tau_arr  = np.array([48.0, 24.0, 72.0, 48.0, 48.0], dtype=np.float64)
    cool_arr = np.array([24.0, 6.0, 24.0, 24.0, 24.0], dtype=np.float64)
    out_fatigue, out_cool_ok = fast_fatigue_cooldown(c_codes, sent_at_hrs, channel_codes, tau_arr, cool_arr)
    df["feat_fatigue"]     = out_fatigue
    df["feat_cooldown_ok"] = out_cool_ok

    for ch_name in ["email", "mobile_push", "web_push"]:
        df[f"channel_{ch_name}"] = (ch_raw_str == ch_name).astype(int)

    for ch_name in ["email", "mobile_push", "sms", "multichannel"]:
        is_ch_mask = (ch_raw_str == ch_name).to_numpy(dtype=bool)
        rec_ch = fast_last_ch_hours(c_codes, sent_at_hrs, is_ch_mask)
        df[f"feat_last_{ch_name}_hours"] = np.nan_to_num(rec_ch, nan=72.0, posinf=72.0, neginf=0.0).clip(0, 8760).astype(np.float32)

    df["feat_last_any_hours"] = df[["feat_last_email_hours", "feat_last_mobile_push_hours"]].min(axis=1).astype(np.float32)
    if "avg_campaign_duration" not in df.columns:
        df["avg_campaign_duration"] = 24.0
    else:
        df["avg_campaign_duration"] = df["avg_campaign_duration"].fillna(24.0).astype(np.float32)

    if "message_type" in df.columns:
        m_type = df["message_type"].fillna("other").astype(str).str.lower()
        for m_name in ["bulk", "transactional", "trigger"]:
            df[f"message_type_{m_name}"] = (m_type == m_name).astype(int)
    else:
        for m_name in ["bulk", "transactional", "trigger"]:
            df[f"message_type_{m_name}"] = 0

    if "email_provider" in df.columns:
        ep = df["email_provider"].fillna("other").astype(str).str.lower().str.strip()
        ALLOWED_EP = {"gmail.com", "mail.ru"}
        ep_red = np.where(ep.isin(ALLOWED_EP), ep, "other")
        df["email_provider_gmail.com"] = (ep_red == "gmail.com").astype(int)
        df["email_provider_mail.ru"]   = (ep_red == "mail.ru").astype(int)
        df["email_provider_other"]     = (ep_red == "other").astype(int)
    else:
        df["email_provider_gmail.com"] = 0
        df["email_provider_mail.ru"]   = 0
        df["email_provider_other"]     = 1

    if "platform" in df.columns:
        p = df["platform"].fillna("other").astype(str).str.lower().str.strip()
        fixed_p = ["desktop", "smartphone", "phablet", "tablet"]
        for name in fixed_p:
            df[f"platform.{name}"] = (p == name).astype(int)
        df["platform."] = (~p.isin(fixed_p)).astype(int)
    else:
        for name in ["desktop", "smartphone", "phablet", "tablet"]:
            df[f"platform.{name}"] = 0
        df["platform."] = 1

    # Real rolling 30-day user behavior metrics: Pure rates & Velocities separated!
    r_op_cnt, r_op_rate, r_cl_rate, r_op_vel, r_cl_vel, r_cad_std = fast_rolling_30d_metrics(c_codes, sent_at_hrs, op_val, cl_val)
    df["u_open_cnt_30d"]         = r_op_cnt
    df["u_open_rate_30d"]        = r_op_rate       # Pure 30-day open rate (0.0 ~ 1.0)
    df["u_click_rate_30d"]       = r_cl_rate       # Pure 30-day click rate (0.0 ~ 1.0)
    df["u_open_velocity_7d_30d"] = r_op_vel        # 7d vs 30d open velocity
    df["u_click_velocity_7d_30d"]= r_cl_vel        # 7d vs 30d click velocity
    df["u_cadence_std_30d"]      = r_cad_std

    # Global 30-day context open rate: must sort by time globally first
    time_sort_idx = np.argsort(sent_at_hrs, kind="mergesort")
    sorted_times = sent_at_hrs[time_sort_idx]
    sorted_ops   = op_val[time_sort_idx]
    ctx_sorted   = fast_ctx_open_rate_30d_sorted(sorted_times, sorted_ops)
    ctx_original = np.empty_like(ctx_sorted)
    ctx_original[time_sort_idx] = ctx_sorted
    df["ctx_tc_open_rate_30d"] = ctx_original

    topic_cols = [c for c in df.columns if str(c).startswith("camp_topic")]
    if len(topic_cols) > 0:
        top_codes = df[topic_cols].fillna(0).to_numpy(dtype=np.int8).argmax(axis=1).astype(np.int32)
    else:
        top_codes = np.zeros(len(df), dtype=np.int32)

    n_topics = int(top_codes.max()) + 1 if len(top_codes) > 0 else 1
    last_top_hrs = fast_client_topic_last_ts(c_codes, top_codes, sent_at_hrs, c_uniques_len, n_topics)
    dt_top = sent_at_hrs - last_top_hrs
    med_top = float(np.nanmedian(dt_top)) if np.isfinite(np.nanmedian(dt_top)) else 48.0
    df["topic_t_since_hours"] = np.nan_to_num(dt_top, nan=med_top, posinf=med_top, neginf=0.0).clip(0, 8760).astype(np.float32)
    df["feat_topic_novelty"]  = np.exp(-df["topic_t_since_hours"].values / 48.0).astype(np.float32)
    df["topic_N7"]            = top_codes.astype(np.int32)

    attr_cols = sorted(set([c for c in df.columns if str(c).startswith(("camp_campaign_type", "camp_channel", "camp_topic"))]))
    if len(attr_cols) > 0:
        vec_mat = df[attr_cols].fillna(0).to_numpy(dtype=np.int8)
    else:
        vec_mat = np.zeros((len(df), 1), dtype=np.int8)
    df["feat_like_last_success"] = fast_like_last_success(c_codes, is_pur, vec_mat, c_uniques_len)
    df["feat_path_align"]        = fast_path_alignment(c_codes, vec_mat, c_uniques_len)


    for col in df.columns:
        if df[col].dtype == object or str(df[col].dtype) == "string":
            df[col] = df[col].astype(str)
        elif "bool" in str(df[col].dtype):
            df[col] = df[col].astype(int)

    print(f"\n[SAVE] Writing final 64-feature dataset to: {OUTPUT_PARQUET}")
    df.to_parquet(OUTPUT_PARQUET, index=False)

    for target_p in [OUTPUT_PARQUET_BASE, OUTPUT_PARQUET_NEW]:
        if target_p != OUTPUT_PARQUET:
            try:
                import shutil
                shutil.copyfile(OUTPUT_PARQUET, target_p)
                print(f"[SAVE] Duplicate copy saved to: {target_p}")
            except Exception:
                pass

    t1 = time.time()
    print("==================================================")
    print(f" SUCCESS! Feature Pipeline Completed in {t1-t0:.2f} seconds!")
    print(f" Output Dataset Shape: {df.shape}")
    print(f" Exact Positive Rate: {df['is_purchased'].mean()*100:.6f}%")
    print("==================================================")

if __name__ == "__main__":
    run_bulletproof_feature_pipeline()


  ZERO-CRASH FAST FEATURE PIPELINE (RAM < 250MB, 35s COMPLETION)
[PATH] Input Extracted Dataset: /mnt/d/LJH/data/messages_extracted_012.parquet
[PATH] Output Parquet File    : /mnt/d/LJH/data/final_data_100k_64.parquet

[FAST SCANNER] Sampling 12,340 Positives + 9,987,660 Negatives directly from Parquet...
  [UNIFORM SAMPLING] Global Negative File Rows: 260,185,373 | Sampling Ratio: 0.038387
[DATASET LOADED IN MEMORY] Total Rows: 9,999,329 | Positives: 12,340 | Negatives: 9,986,989 | Rate: 0.123408%

[FEATURE ENGINEERING] Calculating 64 Causal Domain Features in Memory...

[SAVE] Writing final 64-feature dataset to: /mnt/d/LJH/data/final_data_100k_64.parquet
 SUCCESS! Feature Pipeline Completed in 582.76 seconds!
 Output Dataset Shape: (9999329, 94)
 Exact Positive Rate: 0.123408%


In [2]:
import os, sys, json, gc
import numpy as np
import pandas as pd

PATH = r"/mnt/d/LJH/data/final_data_100k_64.parquet"
if not os.path.exists(PATH):
    PATH = r"D:\LJH\data\final_data_100k_64.parquet"
if not os.path.exists(PATH):
    PATH = r"c:\Users\user\Desktop\마케팅도메인지식기반 구매예측_논문\장바구니 이탈 연구 선행 논문\새 폴더\final_data_64.parquet"

print(f"[DIAGNOSTIC] Loading parquet from: {PATH}")
df = pd.read_parquet(PATH)
if len(df) > 200000:
    print(f"[MEMORY OPTIMIZATION] Subsampling {len(df):,d} rows to 200,000 rows for instant zero-crash correlation calculation...")
    df = df.sample(n=200000, random_state=1).reset_index(drop=True)
TARGET = "is_purchased"

cols_v0 = [
    'avg_campaign_duration', 'avg_time_since_complaint', 'avg_time_since_first_purchase',
    'avg_time_since_last_click', 'avg_time_since_last_open', 'avg_time_since_unsubscribe',
    'camp_campaign_typebulk', 'camp_campaign_typetransactional', 'camp_campaign_typetrigger',
    'camp_channelemail', 'camp_channelmobile_push', 'camp_channelmultichannel', 'camp_channelsms',
    'camp_topicevent', 'camp_topichappy.birthday', 'camp_topicleave.review',
    'camp_topicoffer.after.purchase', 'camp_topicother', 'camp_topicsale.out',
    'channel_email', 'channel_mobile_push', 'channel_web_push',
    'email_provider_gmail.com', 'email_provider_mail.ru', 'email_provider_other',
    'is_holiday',
    'message_type_bulk', 'message_type_transactional', 'message_type_trigger',
    'platform.', 'platform.desktop', 'platform.phablet', 'platform.smartphone', 'platform.tablet',
    'prev_is_clicked', 'prev_is_complained', 'prev_is_opened', 'prev_is_unsubscribed',
    'total_campaigns', 'total_messages', 'total_purchases'
]

cols_v1 = ['days_since_last_purchase', 'feat_rtb_hazard', 'feat_postbuy_refrac']
cols_v2 = ['cal_is_weekend', 'cal_week_of_month', 'feat_dow_shift', 'feat_eoq_bump', 'feat_hour_shift', 'feat_payday_bump']
cols_v3 = ['ctx_tc_open_rate_30d', 'feat_fatigue', 'feat_last_any_hours', 'feat_last_email_hours', 'feat_last_mobile_push_hours', 'u_cadence_std_30d', 'u_click_rate_30d', 'u_open_cnt_30d', 'u_open_rate_30d']
cols_v4 = ['feat_like_last_success', 'feat_path_align', 'feat_topic_novelty', 'topic_N7', 'topic_t_since_hours']

all_domain = cols_v1 + cols_v2 + cols_v3 + cols_v4

print("=" * 70)
print("1. CORRELATION: v1~v4 features vs strongest v0 features")
print("=" * 70)

strong_v0 = ['total_purchases', 'prev_is_clicked', 'prev_is_opened', 'total_campaigns', 'total_messages', 'avg_time_since_first_purchase']

for domain_col in all_domain:
    if domain_col not in df.columns:
        continue
    max_corr = 0
    max_v0 = ""
    for v0_col in strong_v0:
        if v0_col not in df.columns:
            continue
        corr = abs(df[domain_col].corr(df[v0_col]))
        if corr > max_corr:
            max_corr = corr
            max_v0 = v0_col
    flag = "⚠️ HIGH REDUNDANCY" if max_corr > 0.5 else "✅ Unique"
    print(f"  {domain_col:35s} ↔ {max_v0:35s} | corr = {max_corr:.4f} | {flag}")

print("\n" + "=" * 70)
print("2. TARGET CORRELATION: each feature's correlation with is_purchased")
print("=" * 70)

target_corrs = []
for col in cols_v0 + all_domain:
    if col not in df.columns:
        continue
    corr = abs(df[col].corr(df[TARGET]))
    group = "v0" if col in cols_v0 else ("v1" if col in cols_v1 else ("v2" if col in cols_v2 else ("v3" if col in cols_v3 else "v4")))
    target_corrs.append((col, group, corr))

target_corrs.sort(key=lambda x: -x[2])
print(f"\n  Top 20 features by |correlation with {TARGET}|:")
for i, (col, group, corr) in enumerate(target_corrs[:20]):
    marker = "★" if group != "v0" else " "
    print(f"  {i+1:2d}. [{group:2s}] {col:35s} | corr = {corr:.6f} {marker}")

print(f"\n  Bottom 10 (weakest features):")
for i, (col, group, corr) in enumerate(target_corrs[-10:]):
    marker = "★" if group != "v0" else " "
    print(f"  {len(target_corrs)-9+i:2d}. [{group:2s}] {col:35s} | corr = {corr:.6f} {marker}")

print("\n" + "=" * 70)
print("3. BUYER vs NON-BUYER mean comparison for domain features")
print("=" * 70)

buyers = df[df[TARGET] == 1]
non_buyers = df[df[TARGET] == 0].sample(n=min(50000, len(df[df[TARGET]==0])), random_state=1)

for col in all_domain:
    if col not in df.columns:
        continue
    b_mean = buyers[col].mean()
    nb_mean = non_buyers[col].mean()
    diff_pct = abs(b_mean - nb_mean) / max(abs(nb_mean), 1e-10) * 100
    flag = "🔥 BIG DIFF" if diff_pct > 30 else ("📊 Moderate" if diff_pct > 10 else "❌ Tiny diff")
    print(f"  {col:35s} | Buyer: {b_mean:12.4f} | Non-buyer: {nb_mean:12.4f} | Δ = {diff_pct:8.1f}% | {flag}")

print("\n" + "=" * 70)
print("4. VARIANCE: how many domain features have near-zero variance?")
print("=" * 70)

for col in all_domain:
    if col not in df.columns:
        continue
    var = df[col].var()
    nunique = df[col].nunique()
    print(f"  {col:35s} | variance = {var:12.6f} | unique values = {nunique}")


[DIAGNOSTIC] Loading parquet from: /mnt/d/LJH/data/final_data_100k_64.parquet
[MEMORY OPTIMIZATION] Subsampling 9,999,329 rows to 200,000 rows for instant zero-crash correlation calculation...
1. CORRELATION: v1~v4 features vs strongest v0 features
  days_since_last_purchase            ↔ total_purchases                     | corr = 0.5811 | ⚠️ HIGH REDUNDANCY
  feat_rtb_hazard                     ↔ total_purchases                     | corr = 0.1402 | ✅ Unique
  feat_postbuy_refrac                 ↔ total_purchases                     | corr = 0.3014 | ✅ Unique
  cal_is_weekend                      ↔ avg_time_since_first_purchase       | corr = 0.0271 | ✅ Unique
  cal_week_of_month                   ↔ total_purchases                     | corr = 0.0023 | ✅ Unique
  feat_dow_shift                      ↔ total_messages                      | corr = 0.0137 | ✅ Unique
  feat_eoq_bump                       ↔ total_campaigns                     | corr = 0.0175 | ✅ Unique
  feat_hour_shift   